# Level 3A — Simulation Foundation

**Audience:** analysts who understand periodic returns and want to build
reproducible scenario analysis.

**Prerequisites:** Level 1 and basic NumPy/pandas familiarity.

**Learning goals**

1. distinguish a stochastic scenario from a forecast;
2. simulate GBM simple-return and price paths with an explicit random seed;
3. calculate terminal wealth across scenarios;
4. measure floor breaches and cap outcomes without mixing plotting into the
   simulation API.

**Outline:** model assumptions → return paths → price paths → terminal wealth
→ sensitivity exercise.

This notebook independently implements the standard GBM equation. Legacy labs
121–123 were used only to identify the curriculum topic and expected workflow.

## 1. Setup

The example uses monthly steps. `expected_return` is the annual instantaneous
GBM drift μ, while `volatility` is annualized σ.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.simulation import (
    simulate_gbm_prices,
    simulate_gbm_returns,
    terminal_wealth,
    terminal_wealth_stats,
)

## 2. Simulate periodic returns

For \(\Delta t = 1/P\), each simple return is generated from the exact
lognormal step:

\[
r_t = \exp\left((\mu-\tfrac{1}{2}\sigma^2)\Delta t
      + \sigma\sqrt{\Delta t}Z_t\right)-1
\]

`seed` makes the random draw reproducible; it does not make the model more
accurate.

In [ ]:
simulated_returns = simulate_gbm_returns(
    n_years=5,
    n_scenarios=2_000,
    expected_return=0.07,
    volatility=0.15,
    periods_per_year=12,
    seed=42,
)
simulated_returns.iloc[:5, :4]

In [ ]:
pd.Series(
    {
        "rows": simulated_returns.shape[0],
        "scenarios": simulated_returns.shape[1],
        "minimum_period_return": simulated_returns.min().min(),
        "maximum_period_return": simulated_returns.max().max(),
    }
)

## 3. Simulate price paths

Price paths contain an explicit step-zero row. Using the same seed and
parameters makes their periodic percentage changes match the return paths.
Plotting remains notebook presentation logic rather than library behavior.

In [ ]:
prices = simulate_gbm_prices(
    n_years=5,
    n_scenarios=2_000,
    expected_return=0.07,
    volatility=0.15,
    periods_per_year=12,
    initial_price=100.0,
    seed=42,
)
prices.iloc[:, :20].plot(
    legend=False,
    title="Illustrative GBM price scenarios",
    xlabel="Monthly step",
    ylabel="Price index",
    figsize=(9, 4),
)

## 4. Terminal wealth

Treat each scenario column as one possible path. Floor and cap inputs below are
absolute wealth levels in the same unit as `initial_wealth`.

In [ ]:
wealth = terminal_wealth(
    simulated_returns,
    initial_wealth=100.0,
)
wealth.quantile([0.05, 0.25, 0.50, 0.75, 0.95]).rename(
    "terminal_wealth"
)

In [ ]:
terminal_wealth_stats(
    simulated_returns,
    initial_wealth=100.0,
    floor_wealth=80.0,
    cap_wealth=180.0,
)

## 5. Exercise — volatility sensitivity

Hold μ, the horizon, scenario count, and seed constant. Compare annual
volatility of 10% and 25%. Which terminal statistics move most, and why?

In [ ]:
low_vol_returns = simulate_gbm_returns(
    n_years=5,
    n_scenarios=2_000,
    expected_return=0.07,
    volatility=0.10,
    periods_per_year=12,
    seed=7,
)
high_vol_returns = simulate_gbm_returns(
    n_years=5,
    n_scenarios=2_000,
    expected_return=0.07,
    volatility=0.25,
    periods_per_year=12,
    seed=7,
)

# Build and compare two terminal_wealth_stats Series below.

### Answer scaffold

In [ ]:
sensitivity = pd.DataFrame(
    {
        "10% volatility": terminal_wealth_stats(
            low_vol_returns,
            initial_wealth=100.0,
            floor_wealth=80.0,
            cap_wealth=180.0,
        ),
        "25% volatility": terminal_wealth_stats(
            high_vol_returns,
            initial_wealth=100.0,
            floor_wealth=80.0,
            cap_wealth=180.0,
        ),
    }
)
sensitivity.loc[
    [
        "mean",
        "median",
        "standard_deviation",
        "probability_below_floor",
        "expected_shortfall_below_floor",
        "probability_above_cap",
    ]
]

## Interpretation and common pitfalls

- GBM is a model of lognormal diffusion, not a prediction of future prices.
- A fixed seed supports reproducibility; use multiple seeds for implementation
  checks, not to search for a preferred outcome.
- The model assumes constant drift and volatility and independent Gaussian
  shocks. It omits regimes, jumps, fat tails, costs, taxes, and liquidity.
- Keep annual parameters and `periods_per_year` consistent.
- Report the scenario count, horizon, seed policy, assumptions, and thresholds
  alongside terminal statistics.

Next: CPPI can consume scenario returns from this module, but it belongs in a
separate allocation layer with its own floor and rebalancing contracts.